# Quality Score Model v2
### Covers 30+ fruits & vegetables — fresh vs stale classification
### Datasets used:
- `raghavrpotdar/fresh-and-stale-images-of-fruits-and-vegetables` (original)
- `moltean/fruits-360` (Fruits-360 — 131 classes, ~90k images)
- `abdallahalidev/plantvillage-dataset` (PlantVillage — disease/healthy leaves)

### Output: quality_score 1–5
- fresh/healthy → 5
- slightly aged → 3
- stale/diseased → 1


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q kaggle tensorflow pillow opencv-python-headless matplotlib seaborn scikit-learn

In [ ]:
# ── Kaggle credentials ────────────────────────────────────────────────────────
import os
os.environ['KAGGLE_USERNAME'] = 'dhruvbhagchandani19'
os.environ['KAGGLE_KEY']      = 'KGAT_5cccbdc83e00bb08a35f6e2c9378bb1b'
print('Credentials set.')

In [ ]:
# ── Download all three datasets ───────────────────────────────────────────────
!kaggle datasets download -d raghavrpotdar/fresh-and-stale-images-of-fruits-and-vegetables --force
!kaggle datasets download -d moltean/fruits-360 --force
!kaggle datasets download -d abdallahalidev/plantvillage-dataset --force

!unzip -q fresh-and-stale-images-of-fruits-and-vegetables.zip -d ./ds_freshstale
!unzip -q fruits-360.zip                                       -d ./ds_fruits360
!unzip -q plantvillage-dataset.zip                             -d ./ds_plantvillage

print('All datasets downloaded and extracted.')

In [ ]:
# ── Build a unified dataset directory ────────────────────────────────────────
# We map every source class to one of two labels: 'fresh' or 'stale'
# The model predicts fresh/stale; quality_score is derived from confidence.

import os, shutil, pathlib
from tqdm import tqdm

UNIFIED = pathlib.Path('./unified_dataset')
(UNIFIED / 'fresh').mkdir(parents=True, exist_ok=True)
(UNIFIED / 'stale').mkdir(parents=True, exist_ok=True)

def copy_images(src_dir, label, max_per_class=None):
    """Copy images from src_dir into UNIFIED/label."""
    src = pathlib.Path(src_dir)
    if not src.exists():
        print(f'  SKIP (not found): {src}')
        return 0
    exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    files = [f for f in src.rglob('*') if f.suffix.lower() in exts]
    if max_per_class:
        files = files[:max_per_class]
    dest = UNIFIED / label
    for f in tqdm(files, desc=f'{src.name} → {label}', leave=False):
        shutil.copy2(f, dest / f.name)
    return len(files)

total = 0

# ── 1. Fresh-Stale dataset (original) ────────────────────────────────────────
for d in pathlib.Path('./ds_freshstale').iterdir():
    if not d.is_dir(): continue
    label = 'fresh' if d.name.startswith('fresh') else 'stale'
    total += copy_images(d, label)

# ── 2. Fruits-360 ─────────────────────────────────────────────────────────────
# Fruits-360 only has fresh images — all go to 'fresh'
# Limit to 500 per class to keep dataset balanced
fruits360_train = pathlib.Path('./ds_fruits360/fruits-360/Training')
if fruits360_train.exists():
    for cls_dir in fruits360_train.iterdir():
        if cls_dir.is_dir():
            total += copy_images(cls_dir, 'fresh', max_per_class=500)

# ── 3. PlantVillage ───────────────────────────────────────────────────────────
# Classes containing 'healthy' → fresh; everything else → stale
pv_root = pathlib.Path('./ds_plantvillage/plantvillage dataset/color')
if not pv_root.exists():
    pv_root = pathlib.Path('./ds_plantvillage')
for cls_dir in pv_root.rglob('*'):
    if not cls_dir.is_dir(): continue
    label = 'fresh' if 'healthy' in cls_dir.name.lower() else 'stale'
    total += copy_images(cls_dir, label, max_per_class=300)

fresh_count = len(list((UNIFIED / 'fresh').iterdir()))
stale_count = len(list((UNIFIED / 'stale').iterdir()))
print(f'\nUnified dataset ready:')
print(f'  fresh: {fresh_count:,} images')
print(f'  stale: {stale_count:,} images')
print(f'  total: {fresh_count + stale_count:,} images')

In [ ]:
# ── EDA: class balance ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

counts = {'fresh': fresh_count, 'stale': stale_count}
plt.figure(figsize=(6, 4))
plt.bar(counts.keys(), counts.values(), color=['#4CAF50', '#F44336'])
plt.title('Class Distribution')
plt.ylabel('Images')
for k, v in counts.items():
    plt.text(list(counts.keys()).index(k), v + 200, str(v), ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# ── Build tf.data pipelines ───────────────────────────────────────────────────
import tensorflow as tf
import gc

tf.keras.backend.clear_session()
gc.collect()

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
DATA_DIR   = str(UNIFIED)
AUTOTUNE   = tf.data.AUTOTUNE

def preprocess(img, label):
    return tf.keras.applications.mobilenet_v2.preprocess_input(img), label

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation',
    seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

class_names = train_ds.class_names
print('Classes:', class_names)   # should be ['fresh', 'stale']

train_ds = train_ds.map(preprocess).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess).prefetch(AUTOTUNE)

print('Pipelines ready.')

In [ ]:
# ── Model: MobileNetV2 fine-tuned ─────────────────────────────────────────────
base = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, input_shape=(224, 224, 3)
)
base.trainable = False   # freeze base for first phase

x = base.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)
out = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs=base.input, outputs=out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ── Phase 1: train head only (5 epochs) ──────────────────────────────────────
history1 = model.fit(
    train_ds, validation_data=val_ds, epochs=5,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('best_phase1.keras', save_best_only=True)
    ]
)

In [ ]:
# ── Phase 2: unfreeze top 30 layers and fine-tune ─────────────────────────────
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),   # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('best_quality_model.keras', save_best_only=True)
    ]
)

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
loss, acc = model.evaluate(val_ds)
print(f'\nFinal val accuracy: {acc*100:.2f}%')
print(f'Final val loss:     {loss:.4f}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

all_acc  = history1.history['accuracy']  + history2.history['accuracy']
all_val  = history1.history['val_accuracy'] + history2.history['val_accuracy']

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(all_acc,  label='train')
plt.plot(all_val,  label='val')
plt.axvline(len(history1.history['accuracy']) - 1, color='gray', linestyle='--', label='fine-tune start')
plt.title('Accuracy'); plt.legend()

all_loss     = history1.history['loss']     + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
plt.subplot(1, 2, 2)
plt.plot(all_loss,     label='train')
plt.plot(all_val_loss, label='val')
plt.axvline(len(history1.history['loss']) - 1, color='gray', linestyle='--')
plt.title('Loss'); plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Save model + class names ──────────────────────────────────────────────────
import json

model.save('quality_model_v2.keras')

with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

print('Saved: quality_model_v2.keras + class_names.json')

In [ ]:
# ── Quick inference test ──────────────────────────────────────────────────────
import numpy as np, random, pathlib
from PIL import Image as PILImage

def quality_score_from_confidence(predicted_class: str, confidence: float) -> int:
    """
    Maps model output → quality_score 1-5.
    fresh + high confidence → 5
    fresh + low  confidence → 3
    stale + low  confidence → 3
    stale + high confidence → 1
    """
    if predicted_class == 'fresh':
        if confidence >= 0.85: return 5
        if confidence >= 0.65: return 4
        return 3
    else:  # stale
        if confidence >= 0.85: return 1
        if confidence >= 0.65: return 2
        return 3

def predict_image(img_path):
    img = PILImage.open(img_path).resize((224, 224))
    arr = np.array(img)
    if arr.ndim == 2:                    # grayscale → RGB
        arr = np.stack([arr]*3, axis=-1)
    arr = arr[:, :, :3]                  # drop alpha if present
    arr = tf.keras.applications.mobilenet_v2.preprocess_input(arr.astype('float32'))
    preds = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    idx   = int(np.argmax(preds))
    cls   = class_names[idx]
    conf  = float(preds[idx])
    score = quality_score_from_confidence(cls, conf)
    return cls, conf, score

# Test on 5 random images
all_imgs = list(pathlib.Path('./unified_dataset').rglob('*.jpg'))[:]
random.shuffle(all_imgs)
for p in all_imgs[:5]:
    cls, conf, score = predict_image(p)
    print(f'{p.parent.name}/{p.name[:30]:30s}  →  {cls} ({conf*100:.1f}%)  quality={score}/5')

In [ ]:
# ── Download the model ────────────────────────────────────────────────────────
from google.colab import files
files.download('quality_model_v2.keras')
files.download('class_names.json')

---
## Flask server (run this locally after downloading the model)

Save as `ml_/server.py` and run with `python server.py`.

```python
from flask import Flask, request, jsonify
from flask_cors import CORS
import tensorflow as tf
import numpy as np
import json
from PIL import Image
import io

app = Flask(__name__)
CORS(app)

model      = tf.keras.models.load_model('quality_model_v2.keras')
class_names = json.load(open('class_names.json'))

def quality_score(cls, conf):
    if cls == 'fresh':
        if conf >= 0.85: return 5
        if conf >= 0.65: return 4
        return 3
    else:
        if conf >= 0.85: return 1
        if conf >= 0.65: return 2
        return 3

@app.route('/predict', methods=['POST'])
def predict():
    file = request.files.get('file')
    if not file:
        return jsonify({'error': 'No file'}), 400
    img = Image.open(io.BytesIO(file.read())).resize((224, 224)).convert('RGB')
    arr = np.array(img, dtype='float32')
    arr = tf.keras.applications.mobilenet_v2.preprocess_input(arr)
    preds = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    idx   = int(np.argmax(preds))
    cls   = class_names[idx]
    conf  = float(preds[idx])
    return jsonify({
        'filename':         file.filename,
        'predicted_class':  cls,
        'confidence_score': f'{conf*100:.1f}%',
        'quality_score':    quality_score(cls, conf),
        'message':          f'Image classified as {cls} with {conf*100:.1f}% confidence'
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
```
